# HYSPEC SaveMD plotting smoke test

Put your Mantid `SaveMD` `.nxs` file in `../data/hyspec/raw/`, then run this notebook from top to bottom. It checks the file structure, tries to convert a binned MD workspace into `metallix.PointData4D`, and exercises the package plotting helpers.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR = ROOT / "data" / "hyspec"
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

candidates = sorted(RAW_DIR.glob("*.nxs")) + sorted(DATA_DIR.glob("*.nxs"))
NXS_PATH = candidates[0] if candidates else RAW_DIR / "your_hyspec_savemd_file.nxs"

print(f"Project root: {ROOT}")
print(f"Looking for data in: {RAW_DIR}")
print(f"Selected file: {NXS_PATH}")
if not NXS_PATH.exists():
    print("No .nxs file found yet. Put one in data/hyspec/raw/ and re-run this cell.")

## Inspect the NeXus file

This cell is useful even before conversion works. It prints the top HDF5 datasets so we can see how Mantid saved this particular workspace.

In [ ]:
def describe_nexus(path, max_items=120):
    import h5py

    rows = []
    with h5py.File(path, "r") as handle:
        def visit(name, obj):
            if len(rows) >= max_items:
                return
            if isinstance(obj, h5py.Dataset):
                rows.append((name, obj.shape, obj.dtype))
        handle.visititems(visit)
    return rows

if NXS_PATH.exists():
    for name, shape, dtype in describe_nexus(NXS_PATH):
        print(f"{name} | shape={shape} | dtype={dtype}")
else:
    print("Waiting for a .nxs file.")

## Conversion helpers

The preferred path is Mantid `LoadMD`, because `SaveMD` files can store event trees that are not simple rectangular arrays. If Mantid is not available, the fallback only works for files that already contain obvious flat `H`, `K`, `L`, `E`, intensity, and uncertainty arrays.

In [ ]:
import numpy as np

from metallix import PointData4D


def _dimension_centers(dim):
    nbins = int(dim.getNBins())
    lo = float(dim.getMinimum())
    hi = float(dim.getMaximum())
    edges = np.linspace(lo, hi, nbins + 1)
    return 0.5 * (edges[:-1] + edges[1:])


def _dimension_label(dim):
    name = dim.getName() if hasattr(dim, "getName") else ""
    dim_id = dim.getDimensionId() if hasattr(dim, "getDimensionId") else ""
    return f"{name} {dim_id}".lower()


def _infer_dimension_order(dimensions):
    labels = [_dimension_label(dim) for dim in dimensions]
    order = {}
    for i, label in enumerate(labels):
        if "deltae" in label or "energy" in label or "e " in label:
            order["E"] = i
        elif "[h" in label or "h," in label or label.strip().startswith("h"):
            order["H"] = i
        elif "[0,k" in label or "k," in label or label.strip().startswith("k"):
            order["K"] = i
        elif "[0,0,l" in label or "l," in label or label.strip().startswith("l"):
            order["L"] = i

    fallback = ["H", "K", "L", "E"]
    for i, key in enumerate(fallback[: len(dimensions)]):
        order.setdefault(key, i)
    return order


def mdhisto_to_pointdata(workspace):
    signal = np.asarray(workspace.getSignalArray(), dtype=float)
    err2 = np.asarray(workspace.getErrorSquaredArray(), dtype=float)
    sigma = np.sqrt(np.clip(err2, 0.0, np.inf))

    dimensions = [workspace.getDimension(i) for i in range(workspace.getNumDims())]
    axes = [_dimension_centers(dim) for dim in dimensions]
    meshes = np.meshgrid(*axes, indexing="ij")
    order = _infer_dimension_order(dimensions)

    def coord(name):
        if name in order and order[name] < len(meshes):
            return meshes[order[name]].ravel()
        return np.zeros(signal.size, dtype=float)

    return PointData4D(
        H=coord("H"),
        K=coord("K"),
        L=coord("L"),
        E=coord("E"),
        intensity=signal.ravel(),
        sigma=sigma.ravel(),
        metadata={"source": str(NXS_PATH), "loader": "mantid"},
    )


def try_flat_hdf5_pointdata(path):
    import h5py

    aliases = {
        "H": {"h", "[h,0,0]", "qx", "q_sample_x"},
        "K": {"k", "[0,k,0]", "qy", "q_sample_y"},
        "L": {"l", "[0,0,l]", "qz", "q_sample_z"},
        "E": {"e", "deltae", "energy", "energy_transfer"},
        "intensity": {"signal", "intensity", "counts", "y"},
        "sigma": {"sigma", "error", "errors", "uncertainty", "dy"},
    }
    found = {}
    with h5py.File(path, "r") as handle:
        def visit(name, obj):
            if not isinstance(obj, h5py.Dataset):
                return
            leaf = name.rsplit("/", 1)[-1].lower()
            for key, names in aliases.items():
                if key not in found and leaf in names and obj.ndim == 1:
                    found[key] = np.asarray(obj, dtype=float)
        handle.visititems(visit)

    missing = [key for key in aliases if key not in found]
    if missing:
        raise ValueError(f"Could not find flat arrays for: {missing}")
    return PointData4D(metadata={"source": str(path), "loader": "h5py-flat"}, **found)


def load_savemd_pointdata(path, bins=(41, 41, 21, 81)):
    try:
        from mantid.simpleapi import BinMD, LoadMD
    except Exception as exc:
        print(f"Mantid is not available here: {exc}")
        print("Trying the simple flat-HDF5 fallback.")
        return try_flat_hdf5_pointdata(path)

    workspace = LoadMD(Filename=str(path), OutputWorkspace="hyspec_savemd_input")
    if not hasattr(workspace, "getSignalArray"):
        dimensions = [workspace.getDimension(i) for i in range(workspace.getNumDims())]
        aligned_dims = {}
        for i, dim in enumerate(dimensions):
            nbins = bins[min(i, len(bins) - 1)]
            aligned_dims[f"AlignedDim{i}"] = (
                f"{dim.getName()},{float(dim.getMinimum())},{float(dim.getMaximum())},{nbins}"
            )
        workspace = BinMD(
            InputWorkspace=workspace,
            AxisAligned=True,
            OutputWorkspace="hyspec_savemd_binned",
            **aligned_dims,
        )
    return mdhisto_to_pointdata(workspace).valid(require_positive_sigma=False)

## Load the data

In [ ]:
data = None
if NXS_PATH.exists():
    data = load_savemd_pointdata(NXS_PATH)
    print(f"Loaded {data.size:,} points")
    print(f"H range: {np.nanmin(data.H):.4g} to {np.nanmax(data.H):.4g}")
    print(f"K range: {np.nanmin(data.K):.4g} to {np.nanmax(data.K):.4g}")
    print(f"L range: {np.nanmin(data.L):.4g} to {np.nanmax(data.L):.4g}")
    print(f"E range: {np.nanmin(data.E):.4g} to {np.nanmax(data.E):.4g} meV")
else:
    print("Add a .nxs file to data/hyspec/raw/ first.")

## Exercise the plotting tools

In [ ]:
import matplotlib.pyplot as plt

from metallix.plotting import plot_2d_map, plot_energy_cut, plot_q_cut


def window_cut(data, *, center=None, half_width=None, max_points=20_000):
    if center is None:
        center = {
            "H": float(np.nanmedian(data.H)),
            "K": float(np.nanmedian(data.K)),
            "L": float(np.nanmedian(data.L)),
            "E": float(np.nanmedian(data.E)),
        }
    if half_width is None:
        half_width = {"H": 0.08, "K": 0.08, "L": 0.08, "E": 2.0}

    mask = np.ones(data.size, dtype=bool)
    for name in ("H", "K", "L", "E"):
        values = getattr(data, name)
        mask &= np.abs(values - center[name]) <= half_width[name]

    idx = np.flatnonzero(mask)
    if idx.size == 0:
        finite = np.flatnonzero(data.valid_mask(require_positive_sigma=False))
        idx = finite[:max_points]
    elif idx.size > max_points:
        idx = idx[:: max(1, idx.size // max_points)][:max_points]

    temp = data.temperature[idx] if isinstance(data.temperature, np.ndarray) else data.temperature
    return PointData4D(
        H=data.H[idx],
        K=data.K[idx],
        L=data.L[idx],
        E=data.E[idx],
        intensity=data.intensity[idx],
        sigma=data.sigma[idx],
        temperature=temp,
        metadata=dict(data.metadata),
    )


if data is None:
    print("No data loaded yet.")
else:
    ecut = window_cut(
        data,
        half_width={"H": 0.08, "K": 0.08, "L": 0.08, "E": np.inf},
        max_points=5000,
    )
    qcut = window_cut(
        data,
        half_width={"H": np.inf, "K": 0.08, "L": 0.08, "E": 2.0},
        max_points=5000,
    )
    emap = window_cut(
        data,
        half_width={"H": np.inf, "K": np.inf, "L": 0.08, "E": 2.0},
        max_points=20000,
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    plot_energy_cut(ecut, ax=axes[0])
    axes[0].set_title("Energy cut")
    plot_q_cut(qcut, "H", ax=axes[1])
    axes[1].set_title("H cut")
    plot_2d_map(emap.H, emap.K, emap.intensity, ax=axes[2], xlabel="H (RLU)", ylabel="K (RLU)")
    axes[2].set_title("H-K map")
    plt.show()